In [0]:
from pyspark.sql.functions import col

# 1. Configuración de Parámetros
dbutils.widgets.dropdown("env", "dev", ["dev", "prod"], "Ambiente a limpiar")
env = dbutils.widgets.get("env")
catalog = f"{env}_fraud"

# 2. Definir esquemas protegidos (Los que NO se deben borrar)
# Agregamos 'default' e 'information_schema' por seguridad del sistema
protected_schemas = ["raw", "default", "information_schema"]

print(f"⚠️ Iniciando limpieza profunda en el catálogo: {catalog}")
print(f"🛡️ Esquemas protegidos: {protected_schemas}\n")

# 3. Listar todos los esquemas del catálogo
schemas_df = spark.sql(f"SHOW SCHEMAS IN {catalog}")
all_schemas = [row.databaseName for row in schemas_df.collect()]

# 4. Lógica de Eliminación
dropped_count = 0

for schema in all_schemas:
    if schema.lower() not in protected_schemas:
        try:
            # CASCADE borra tablas, vistas y funciones dentro del esquema
            spark.sql(f"DROP SCHEMA IF EXISTS {catalog}.{schema} CASCADE")
            print(f"✅ Esquema eliminado: {schema}")
            dropped_count += 1
        except Exception as e:
            print(f"❌ Error al eliminar {schema}: {e}")
    else:
        print(f"🔒 Saltando esquema protegido: {schema}")

# 5. Resumen final
print("\n" + "═" * 50)
print(f" ✨ LIMPIEZA COMPLETADA ".center(50, "═"))
print("═" * 50)
print(f"Esquemas eliminados: {dropped_count}")
print(f"Catálogo actual: {catalog}")
display(spark.sql(f"SHOW SCHEMAS IN {catalog}"))